In [61]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mtick
import warnings
from math import factorial

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 220)

# ── Parámetros globales del modelo ───────────────────────────────────────────
WQ_MAX_MIN  = 5.0    # Wq máximo aceptable (minutos)
RHO_MAX     = 0.85   # Utilización máxima operativa
MAX_SERVERS = 15     # Tope de búsqueda en optimización

# ── Paleta de colores por estado ─────────────────────────────────────────────
COLOR_MAP = {
    'Óptimo':           '#2ecc71',
    'Sobredimensionado': '#3498db',
    'Subdimensionado':   '#e74c3c',
    'Crítico':           '#8e44ad'
}

# ── Estilo visual ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f172a',
    'axes.facecolor':   '#1e293b',
    'axes.edgecolor':   '#334155',
    'axes.labelcolor':  '#94a3b8',
    'xtick.color':      '#64748b',
    'ytick.color':      '#64748b',
    'text.color':       '#e2e8f0',
    'grid.color':       '#1e293b',
    'grid.linestyle':   '--',
    'grid.alpha':       0.4,
    'axes.titlecolor':  '#f1f5f9',
    'axes.titlesize':   12,
    'axes.titleweight': 'bold',
    'legend.facecolor': '#1e293b',
    'legend.edgecolor': '#334155',
    'legend.labelcolor':'#94a3b8',
    'font.family':      'monospace',
})

print('✅ Configuración cargada')
print(f'   Wq_max = {WQ_MAX_MIN} min  |  ρ_max = {RHO_MAX}  |  s_max = {MAX_SERVERS}')

✅ Configuración cargada
   Wq_max = 5.0 min  |  ρ_max = 0.85  |  s_max = 15


In [62]:
df   = pd.read_csv('df.csv')
dist = pd.read_csv('dist_agencias.csv')
cols_horas = [c for c in dist.columns if ':' in c]
agencias   = df['agencia'].unique()

print(f'✅ df.csv:   {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'✅ dist.csv: {dist.shape[0]} filas × {len(cols_horas)} franjas horarias')
print(f'   Agencias: {", ".join(agencias)}')
print()
print(df.to_string(index=False))

✅ df.csv:   30 filas × 6 columnas
✅ dist.csv: 18 filas × 10 franjas horarias
   Agencias: AG.PARQUE DE LAS FLORES LOS GUINDALES, AG. CIUDAD UNIVERSITARIA, AG. OPEN SAN CARLOS

                              agencia                hora  lambda  servidores      Cs      Cq
AG.PARQUE DE LAS FLORES LOS GUINDALES 09:00:00 - 09:59:59 50.5000           5 13.6400 27.2900
AG.PARQUE DE LAS FLORES LOS GUINDALES 10:00:00 - 10:59:59 60.1200           5 13.6400 27.2900
AG.PARQUE DE LAS FLORES LOS GUINDALES 11:00:00 - 11:59:59 58.9000           5 13.6400 27.2900
AG.PARQUE DE LAS FLORES LOS GUINDALES 12:00:00 - 12:59:59 43.4800           5 13.6400 27.2900
AG.PARQUE DE LAS FLORES LOS GUINDALES 13:00:00 - 13:59:59 35.0000           5 13.6400 27.2900
AG.PARQUE DE LAS FLORES LOS GUINDALES 14:00:00 - 14:59:59 31.6400           5 13.6400 27.2900
AG.PARQUE DE LAS FLORES LOS GUINDALES 15:00:00 - 15:59:59 44.2300           5 13.6400 27.2900
AG.PARQUE DE LAS FLORES LOS GUINDALES 16:00:00 - 16:59:59 59.1100       

In [63]:
# Tiempo promedio de servicio por tipo de operación (minutos)
T_SERVICIO = {
    'DEPOSITO':                         3.2,
    'COBRANZA LOCALES OTRAS AGENCIAS':  2.8,
    'COBRANZAS':                        2.8,
    'RETIRO':                           3.0,
    'DESEMBOLSO':                      18.5,
    'APERTURA':                        12.0
}

dist['t_s'] = dist['tipo_ope'].map(T_SERVICIO)


def calcular_mu(df_base, dist_df, cols_horas):
    """
    Calcula μ ponderado por la mezcla de operaciones de cada (agencia, hora).
    Reutilizado del Modelo v4 — sin modificación.
    """
    rows = []
    for _, row in df_base.iterrows():
        agencia = row['agencia']
        hora    = row['hora']
        d       = dist_df[dist_df['agencia'] == agencia]

        if hora not in d.columns:
            rows.append({'agencia': agencia, 'hora': hora, 'mu': np.nan})
            continue

        probs = d[hora].values.astype(float)
        ts    = d['t_s'].values.astype(float)
        total = probs.sum()

        if total == 0 or np.isnan(total):
            rows.append({'agencia': agencia, 'hora': hora, 'mu': np.nan})
            continue

        E_T = np.dot(probs, ts) / total
        mu  = 60.0 / E_T if E_T > 0 else np.nan
        rows.append({'agencia': agencia, 'hora': hora, 'mu': mu})

    return pd.DataFrame(rows)


mu_df = calcular_mu(df, dist, cols_horas)
df2   = df.merge(mu_df, on=['agencia', 'hora'], how='left')

print('μ calculado por agencia-hora:')
print(df2[['agencia', 'hora', 'lambda', 'mu', 'servidores']].to_string(index=False))

μ calculado por agencia-hora:
                              agencia                hora  lambda      mu  servidores
AG.PARQUE DE LAS FLORES LOS GUINDALES 09:00:00 - 09:59:59 50.5000 15.6454           5
AG.PARQUE DE LAS FLORES LOS GUINDALES 10:00:00 - 10:59:59 60.1200 15.0417           5
AG.PARQUE DE LAS FLORES LOS GUINDALES 11:00:00 - 11:59:59 58.9000 14.8570           5
AG.PARQUE DE LAS FLORES LOS GUINDALES 12:00:00 - 12:59:59 43.4800 14.2323           5
AG.PARQUE DE LAS FLORES LOS GUINDALES 13:00:00 - 13:59:59 35.0000 14.4637           5
AG.PARQUE DE LAS FLORES LOS GUINDALES 14:00:00 - 14:59:59 31.6400 14.7027           5
AG.PARQUE DE LAS FLORES LOS GUINDALES 15:00:00 - 15:59:59 44.2300 15.1287           5
AG.PARQUE DE LAS FLORES LOS GUINDALES 16:00:00 - 16:59:59 59.1100 14.9212           5
AG.PARQUE DE LAS FLORES LOS GUINDALES 17:00:00 - 17:59:59 65.2800 13.4813           5
AG.PARQUE DE LAS FLORES LOS GUINDALES 18:00:00 - 18:59:59 23.4500 10.8063           5
             AG. CIUDAD 

In [64]:
def mms_metricas(lam, mu, s, Cs=0, Cq=0):
    """
    ══════════════════════════════════════════════════════════════════
    MODELO M/M/s — REUTILIZADO DEL MODELO v4 (sin modificación)
    ══════════════════════════════════════════════════════════════════
    Parámetros:
      lam : tasa de llegadas (clientes/hora)
      mu  : tasa de servicio por servidor (clientes/hora)
      s   : número de servidores
      Cs  : costo por servidor activo (S/./hora)
      Cq  : costo de espera por cliente (S/./hora)
    Retorna:
      dict con todas las métricas, o None si parámetros inválidos
    """
    nan_r = dict(rho=np.nan, P0=np.nan, Lq=np.nan, Wq_h=np.nan,
                 Ws_h=np.nan, Ls=np.nan, Wq_m=np.nan, Ws_m=np.nan,
                 serv_ocupados=np.nan, serv_ociosos=np.nan, CT=np.nan, valido=False)

    if mu <= 0 or s <= 0 or lam < 0:
        return nan_r

    s   = int(s)
    rho = lam / (s * mu)
    r   = lam / mu

    if rho >= 1:
        return {**nan_r, 'rho': rho, 'valido': False}

    # Probabilidad de sistema vacío
    sum_k  = sum(r**k / factorial(k) for k in range(s))
    term_s = (r**s) / (factorial(s) * (1 - rho))
    P0     = 1.0 / (sum_k + term_s)

    # Métricas de cola
    Lq   = P0 * (r**s) * rho / (factorial(s) * (1 - rho)**2)
    Wq_h = Lq / lam if lam > 0 else 0.0
    Ws_h = Wq_h + 1.0 / mu
    Ls   = lam * Ws_h
    Wq_m = Wq_h * 60
    Ws_m = Ws_h * 60

    # Servidores
    serv_ocupados = Ls - Lq
    serv_ociosos  = s - serv_ocupados

    # Costo total
    CT = (Cs + Cq) * serv_ocupados + Cs * max(serv_ociosos, 0) + lam * Cq * Wq_h

    return dict(
        rho=rho, P0=P0, Lq=Lq, Wq_h=Wq_h, Ws_h=Ws_h,
        Ls=Ls, Wq_m=Wq_m, Ws_m=Ws_m,
        serv_ocupados=serv_ocupados, serv_ociosos=serv_ociosos,
        CT=CT, valido=True
    )


# ── Test de verificación ─────────────────────────────────────────────────────
t = mms_metricas(60, 20, 5, 13.64, 27.29)
print('Test M/M/s (λ=60, μ=20, s=5):')
print(f'  ρ     = {t["rho"]:.4f}')
print(f'  Wq    = {t["Wq_m"]:.4f} min')
print(f'  CT    = S/.{t["CT"]:.2f}')
print(f'  Válido= {t["valido"]}')
print()
print('✅ Modelo M/M/s listo (reutilizado del v4 sin modificación)')

Test M/M/s (λ=60, μ=20, s=5):
  ρ     = 0.6000
  Wq    = 0.3542 min
  CT    = S/.159.74
  Válido= True

✅ Modelo M/M/s listo (reutilizado del v4 sin modificación)


In [65]:
def optimizar_servidores(lam, mu, Cs, Cq, s_max=MAX_SERVERS):
    """
    Capa de optimización sobre el modelo M/M/s existente.
    Busca el mínimo s que cumple: ρ < RHO_MAX y Wq ≤ WQ_MAX_MIN.

    Retorna:
      (s_opt, metricas_opt)  si existe solución factible
      None                   si no existe (caso Crítico)
    """
    if np.isnan(mu) or mu <= 0 or np.isnan(lam) or lam <= 0:
        return None

    # s mínimo para estabilidad del sistema (ρ < 1)
    s_min = max(1, int(np.ceil(lam / mu)) + 1)

    for s in range(s_min, s_max + 1):
        m = mms_metricas(lam, mu, s, Cs, Cq)
        if not m['valido']:
            continue
        # ← Verificar ambas restricciones
        if m['rho'] < RHO_MAX and m['Wq_m'] <= WQ_MAX_MIN:
            return s, m

    return None  # No existe solución factible


def clasificar_estado(row):
    """
    Clasifica el estado operativo comparando s_actual vs s_optimo.

    Estados:
      Óptimo           → s_actual == s_optimo Y cumple restricciones
      Sobredimensionado → s_actual > s_optimo  (exceso de personal, sobrecosto)
      Subdimensionado  → s_actual < s_optimo  (déficit, mala calidad de servicio)
      Crítico          → no existe s factible  (sistema colapsa)
    """
    if pd.isna(row['s_optimo']):
        return 'Crítico'
    diff = row['s_actual'] - row['s_optimo']
    if diff > 0:
        return 'Sobredimensionado'
    elif diff < 0:
        return 'Subdimensionado'
    else:
        return 'Óptimo'


print('✅ Capa de optimización definida')
print(f'   Búsqueda: s ∈ [s_min, {MAX_SERVERS}]')
print(f'   Criterio: ρ < {RHO_MAX}  AND  Wq ≤ {WQ_MAX_MIN} min')

✅ Capa de optimización definida
   Búsqueda: s ∈ [s_min, 15]
   Criterio: ρ < 0.85  AND  Wq ≤ 5.0 min


In [66]:
registros = []

for _, row in df2.iterrows():
    lam   = row['lambda']
    mu    = row['mu']
    s_act = int(row['servidores'])
    Cs    = row['Cs']
    Cq    = row['Cq']
    hora  = row['hora'][:5]
    agencia = row['agencia']

    # ── Estado actual con el M/M/s existente ─────────────────────────────────
    m_act   = mms_metricas(lam, mu, s_act, Cs, Cq)
    rho_act = m_act['rho']   if m_act['valido'] else np.nan
    Wq_act  = m_act['Wq_m']  if m_act['valido'] else np.nan
    CT_act  = m_act['CT']    if m_act['valido'] else np.nan

    # ── Optimización: mínimo s que cumple restricciones ──────────────────────
    resultado = optimizar_servidores(lam, mu, Cs, Cq)

    if resultado is not None:
        s_opt, m_opt = resultado
        rho_opt = m_opt['rho']
        Wq_opt  = m_opt['Wq_m']
        CT_opt  = m_opt['CT']
    else:
        s_opt = np.nan
        rho_opt = np.nan
        Wq_opt  = np.nan
        CT_opt  = np.nan

    # ── Métricas derivadas ────────────────────────────────────────────────────
    diff_s     = (s_act - s_opt) if not np.isnan(s_opt) else np.nan
    ahorro_CT  = (CT_act - CT_opt) if not np.isnan(CT_opt) else np.nan
    cap_actual = s_act * mu
    cap_opt    = s_opt * mu if not np.isnan(s_opt) else np.nan

    registros.append(dict(
        agencia              = agencia,
        hora                 = hora,
        lambda_              = lam,
        mu                   = round(mu, 3),
        s_actual             = s_act,
        s_optimo             = int(s_opt) if not np.isnan(s_opt) else np.nan,
        rho_actual           = round(rho_act, 4) if not np.isnan(rho_act) else np.nan,
        rho_optimo           = round(rho_opt, 4) if not np.isnan(rho_opt) else np.nan,
        Wq_actual            = round(Wq_act, 4)  if not np.isnan(Wq_act)  else np.nan,
        Wq_optimo            = round(Wq_opt, 4)  if not np.isnan(Wq_opt)  else np.nan,
        CT_actual            = round(CT_act, 2)   if not np.isnan(CT_act)  else np.nan,
        CT_optimo            = round(CT_opt, 2)   if not np.isnan(CT_opt)  else np.nan,
        diferencia_servidores= int(diff_s)  if not np.isnan(diff_s) else np.nan,
        ahorro_costo         = round(ahorro_CT, 2) if not np.isnan(ahorro_CT) else np.nan,
        capacidad_actual     = round(cap_actual, 2),
        capacidad_optima     = round(cap_opt, 2) if not np.isnan(cap_opt) else np.nan,
        cumple_rho_actual    = 'SÍ' if (not np.isnan(rho_act) and rho_act < RHO_MAX) else 'NO',
        cumple_Wq_actual     = 'SÍ' if (not np.isnan(Wq_act) and Wq_act <= WQ_MAX_MIN) else 'NO',
    ))

df_final = pd.DataFrame(registros)

# Asignar estado operativo
df_final['estado'] = df_final.apply(clasificar_estado, axis=1)

# Asignar turno
def asignar_turno(hora):
    return 'Turno 1 (09-14)' if int(hora[:2]) < 14 else 'Turno 2 (14-18)'

df_final['turno'] = df_final['hora'].apply(asignar_turno)

df_final

# print('✅ DataFrame final construido:', df_final.shape)
# print()
# print(df_final[['agencia','hora','lambda_','mu','s_actual','s_optimo',
#                 'rho_actual','Wq_actual','diferencia_servidores','estado']].to_string(index=False))

,agencia,hora,lambda_,mu,s_actual,s_optimo,rho_actual,rho_optimo,Wq_actual,Wq_optimo,CT_actual,CT_optimo,diferencia_servidores,ahorro_costo,capacidad_actual,capacidad_optima,cumple_rho_actual,cumple_Wq_actual,estado,turno
0,AG.PARQUE DE LAS FLORES LOS GUINDALES,09:00,50.5000,15.6450,5,5,0.6456,0.6456,0.6412,0.6412,171.0100,171.0100,0,0.0000,78.2300,78.2300,SÍ,SÍ,Óptimo,Turno 1 (09-14)
1,AG.PARQUE DE LAS FLORES LOS GUINDALES,10:00,60.1200,15.0420,5,5,0.7994,0.7994,2.1987,2.1987,237.4000,237.4000,0,0.0000,75.2100,75.2100,SÍ,SÍ,Óptimo,Turno 1 (09-14)
2,AG.PARQUE DE LAS FLORES LOS GUINDALES,11:00,58.9000,14.8570,5,5,0.7929,0.7929,2.1078,2.1078,232.8600,232.8600,0,0.0000,74.2900,74.2900,SÍ,SÍ,Óptimo,Turno 1 (09-14)
3,AG.PARQUE DE LAS FLORES LOS GUINDALES,12:00,43.4800,14.2320,5,5,0.6110,0.6110,0.5419,0.5419,162.2900,162.2900,0,0.0000,71.1600,71.1600,SÍ,SÍ,Óptimo,Turno 1 (09-14)
4,AG.PARQUE DE LAS FLORES LOS GUINDALES,13:00,35.0000,14.4640,5,4,0.4840,0.6050,0.1877,0.7703,137.2300,132.8600,1,4.3700,72.3200,57.8500,SÍ,SÍ,Sobredimensionado,Turno 1 (09-14)
5,AG.PARQUE DE LAS FLORES LOS GUINDALES,14:00,31.6400,14.7030,5,4,0.4304,0.5380,0.1112,0.4713,128.5300,120.0700,1,8.4600,73.5100,58.8100,SÍ,SÍ,Sobredimensionado,Turno 2 (14-18)
6,AG.PARQUE DE LAS FLORES LOS GUINDALES,15:00,44.2300,15.1290,5,4,0.5847,0.7309,0.4157,1.7605,156.3500,169.7600,1,-13.4100,75.6400,60.5100,SÍ,SÍ,Sobredimensionado,Turno 2 (14-18)
7,AG.PARQUE DE LAS FLORES LOS GUINDALES,16:00,59.1100,14.9210,5,5,0.7923,0.7923,2.0883,2.0883,232.4500,232.4500,0,0.0000,74.6100,74.6100,SÍ,SÍ,Óptimo,Turno 2 (14-18)
8,AG.PARQUE DE LAS FLORES LOS GUINDALES,17:00,65.2800,13.4810,5,6,0.9685,0.8070,26.0192,2.0457,972.9000,274.7300,-1,698.1700,67.4100,80.8900,NO,NO,Subdimensionado,Turno 2 (14-18)
9,AG.PARQUE DE LAS FLORES LOS GUINDALES,18:00,23.4500,10.8060,5,4,0.4340,0.5425,0.1569,0.6627,129.0900,120.8500,1,8.2500,54.0300,43.2300,SÍ,SÍ,Sobredimensionado,Turno 2 (14-18)


In [67]:
total = len(df_final)
dist_estado = df_final['estado'].value_counts()
pct_cumple_rho = (df_final['cumple_rho_actual'] == 'SÍ').mean() * 100
pct_cumple_wq  = (df_final['cumple_Wq_actual']  == 'SÍ').mean() * 100
ahorro_total   = df_final['ahorro_costo'].sum()
exceso_total   = df_final.loc[df_final['diferencia_servidores']>0, 'diferencia_servidores'].sum()
deficit_total  = abs(df_final.loc[df_final['diferencia_servidores']<0, 'diferencia_servidores'].sum())

print('═' * 60)
print('          RESUMEN EJECUTIVO — MVP OPTIMIZACIÓN')
print('═' * 60)
print(f'  Franjas horarias analizadas : {total}')
print(f'  Agencias                    : {df_final["agencia"].nunique()}')
print()
print('  DISTRIBUCIÓN DE ESTADOS:')
for estado, cnt in dist_estado.items():
    pct = cnt/total*100
    print(f'    {estado:<22}: {cnt:2d} franjas ({pct:.0f}%)')
print()
print(f'  CUMPLIMIENTO SLA ρ < 0.85  : {pct_cumple_rho:.0f}%')
print(f'  CUMPLIMIENTO SLA Wq ≤ 5min  : {pct_cumple_wq:.0f}%')
print()
print(f'  Exceso de servidores (total): {exceso_total} puestos')
print(f'  Déficit de servidores (tot) : {deficit_total} puestos')
print(f'  Ahorro potencial total      : S/.{ahorro_total:.2f}/hora')
print('═' * 60)

════════════════════════════════════════════════════════════
          RESUMEN EJECUTIVO — MVP OPTIMIZACIÓN
════════════════════════════════════════════════════════════
  Franjas horarias analizadas : 30
  Agencias                    : 3

  DISTRIBUCIÓN DE ESTADOS:
    Sobredimensionado     : 14 franjas (47%)
    Óptimo                : 13 franjas (43%)
    Subdimensionado       :  3 franjas (10%)

  CUMPLIMIENTO SLA ρ < 0.85  : 90%
  CUMPLIMIENTO SLA Wq ≤ 5min  : 93%

  Exceso de servidores (total): 17 puestos
  Déficit de servidores (tot) : 3 puestos
  Ahorro potencial total      : S/.1287.20/hora
════════════════════════════════════════════════════════════


In [68]:
df_final

,agencia,hora,lambda_,mu,s_actual,s_optimo,rho_actual,rho_optimo,Wq_actual,Wq_optimo,CT_actual,CT_optimo,diferencia_servidores,ahorro_costo,capacidad_actual,capacidad_optima,cumple_rho_actual,cumple_Wq_actual,estado,turno
0,AG.PARQUE DE LAS FLORES LOS GUINDALES,09:00,50.5000,15.6450,5,5,0.6456,0.6456,0.6412,0.6412,171.0100,171.0100,0,0.0000,78.2300,78.2300,SÍ,SÍ,Óptimo,Turno 1 (09-14)
1,AG.PARQUE DE LAS FLORES LOS GUINDALES,10:00,60.1200,15.0420,5,5,0.7994,0.7994,2.1987,2.1987,237.4000,237.4000,0,0.0000,75.2100,75.2100,SÍ,SÍ,Óptimo,Turno 1 (09-14)
2,AG.PARQUE DE LAS FLORES LOS GUINDALES,11:00,58.9000,14.8570,5,5,0.7929,0.7929,2.1078,2.1078,232.8600,232.8600,0,0.0000,74.2900,74.2900,SÍ,SÍ,Óptimo,Turno 1 (09-14)
3,AG.PARQUE DE LAS FLORES LOS GUINDALES,12:00,43.4800,14.2320,5,5,0.6110,0.6110,0.5419,0.5419,162.2900,162.2900,0,0.0000,71.1600,71.1600,SÍ,SÍ,Óptimo,Turno 1 (09-14)
4,AG.PARQUE DE LAS FLORES LOS GUINDALES,13:00,35.0000,14.4640,5,4,0.4840,0.6050,0.1877,0.7703,137.2300,132.8600,1,4.3700,72.3200,57.8500,SÍ,SÍ,Sobredimensionado,Turno 1 (09-14)
5,AG.PARQUE DE LAS FLORES LOS GUINDALES,14:00,31.6400,14.7030,5,4,0.4304,0.5380,0.1112,0.4713,128.5300,120.0700,1,8.4600,73.5100,58.8100,SÍ,SÍ,Sobredimensionado,Turno 2 (14-18)
6,AG.PARQUE DE LAS FLORES LOS GUINDALES,15:00,44.2300,15.1290,5,4,0.5847,0.7309,0.4157,1.7605,156.3500,169.7600,1,-13.4100,75.6400,60.5100,SÍ,SÍ,Sobredimensionado,Turno 2 (14-18)
7,AG.PARQUE DE LAS FLORES LOS GUINDALES,16:00,59.1100,14.9210,5,5,0.7923,0.7923,2.0883,2.0883,232.4500,232.4500,0,0.0000,74.6100,74.6100,SÍ,SÍ,Óptimo,Turno 2 (14-18)
8,AG.PARQUE DE LAS FLORES LOS GUINDALES,17:00,65.2800,13.4810,5,6,0.9685,0.8070,26.0192,2.0457,972.9000,274.7300,-1,698.1700,67.4100,80.8900,NO,NO,Subdimensionado,Turno 2 (14-18)
9,AG.PARQUE DE LAS FLORES LOS GUINDALES,18:00,23.4500,10.8060,5,4,0.4340,0.5425,0.1569,0.6627,129.0900,120.8500,1,8.2500,54.0300,43.2300,SÍ,SÍ,Sobredimensionado,Turno 2 (14-18)


In [ ]:
turno_rows = []

for (agencia, turno), grp in df_final.groupby(['agencia', 'turno']):
    lam_max = grp['lambda_'].max()
    lam_prom = grp['lambda_'].mean()
    mu_prom = grp['mu'].mean()


    s_opt_turno = grp['s_optimo'].max() if grp['s_optimo'].notna().any() else np.nan
    s_act_turno = grp['s_actual'].iloc[0]

    n_sobre = (grp['estado'] == 'Sobredimensionado').sum()
    n_sub   = (grp['estado'] == 'Subdimensionado').sum()
    n_crit  = (grp['estado'] == 'Crítico').sum()
    n_opt   = (grp['estado'] == 'Óptimo').sum()
    
    wq_act = grp['Wq_actual'].max()
    wq_opt = grp['Wq_optimo'].mean()
    rho_act   = grp['rho_actual'].max()
    rho_opt   = grp['rho_optimo'].mean()
    # ahorro_t   = grp['ahorro_costo'].sum()
    ahorro_t = max(0, grp['ahorro_costo'].fillna(0).sum())

    pct_cumple = ((grp['cumple_Wq_actual'] == 'SÍ') & (grp['cumple_rho_actual'] == 'SÍ')).mean() * 100
    
    diff_t = s_act_turno - s_opt_turno if not np.isnan(s_opt_turno) else np.nan
    if pd.isna(diff_t):       decision = '⚠️ CRÍTICO'
    elif diff_t > 0:          decision = '🟢 REDUCIR'
    elif diff_t < 0:          decision = '🔴 CONTRATAR'
    else:                     decision = '🟡 MANTENER'   


    turno_rows.append(dict(
        agencia=agencia, turno=turno,
        n_horas=len(grp),
        lambda_max=round(lam_max, 2),
        lambda_prom=round(lam_prom, 2),
        mu_prom=round(mu_prom, 3),
        s_actual=int(s_act_turno),
        s_optimo_turno=int(s_opt_turno) if not np.isnan(s_opt_turno) else np.nan,
        diferencia=int(diff_t) if not np.isnan(diff_t) else np.nan,
        rho_act=round(rho_act, 3),
        rho_opt=round(rho_opt, 3),
        Wq_act=round(wq_act, 3),
        Wq_opt=round(wq_opt, 3),
        pct_cumplimiento=round(pct_cumple, 1),
        horas_optimas=int(n_opt),
        horas_sobre=int(n_sobre),
        horas_sub=int(n_sub),
        horas_criticas=int(n_crit),
        ahorro_turno=round(ahorro_t, 2),
        decision=decision
    ))

df_turnos = pd.DataFrame(turno_rows)
df_turnos

,agencia,turno,n_horas,lambda_max,lambda_prom,mu_prom,s_actual,s_optimo_turno,rho_act,rho_opt,Wq_act,Wq_opt,pct_cumplimiento,horas_optimas,horas_sobre,horas_sub,horas_criticas,ahorro_turno,decision
0,AG. CIUDAD UNIVERSITARIA,Turno 1 (09-14),5,65.6200,53.1700,14.6590,6,6,0.7320,0.6910,0.9750,0.9460,100.0000,2,3,0,0,7.4700,🟡 MANTENER
1,AG. CIUDAD UNIVERSITARIA,Turno 2 (14-18),5,67.8300,46.7900,13.4650,6,7,0.8700,0.6490,3.9810,0.7880,80.0000,1,3,1,0,110.5100,🔴 CONTRATAR
2,AG. OPEN SAN CARLOS,Turno 1 (09-14),5,36.9200,29.8800,13.0830,4,4,0.6930,0.6340,1.5330,1.3200,100.0000,3,2,0,0,0.0000,🟡 MANTENER
3,AG. OPEN SAN CARLOS,Turno 2 (14-18),5,46.2900,29.6900,12.3230,4,5,0.9540,0.6260,24.2510,1.3260,80.0000,2,2,1,0,465.2200,🔴 CONTRATAR
4,AG.PARQUE DE LAS FLORES LOS GUINDALES,Turno 1 (09-14),5,60.1200,49.6000,14.8480,5,5,0.7990,0.6910,2.1990,1.2520,100.0000,4,1,0,0,4.3700,🟡 MANTENER
5,AG.PARQUE DE LAS FLORES LOS GUINDALES,Turno 2 (14-18),5,65.2800,44.7400,13.8080,5,6,0.9680,0.6820,26.0190,1.4060,80.0000,1,3,1,0,701.4700,🔴 CONTRATAR
